# 03 — Silver MERGE Upsert

**Project:** Incremental Orders Pipeline with Delta Lake  
**Layer:** Silver  
**Source:** `orders_project.bronze_orders_raw`  
**Target:** `orders_project.silver_orders_current`

This notebook creates and updates the Silver current-state orders table using Delta Lake `MERGE INTO`.

The Bronze table stores all raw order events.  
The Silver table stores only the latest known state of each order.

This notebook demonstrates:

- Incremental processing
- Upserts
- Delta Lake `MERGE INTO`
- Deduplication by `order_id`
- Keeping the latest event based on `event_ts`
- Idempotent processing

In [0]:
from pyspark.sql.functions import col, current_timestamp, row_number
from pyspark.sql.window import Window

In [0]:
spark.sql("USE SCHEMA orders_project")

In [0]:
dbutils.widgets.dropdown(
    "batch_id",
    "1",
    ["1", "2", "3"],
    "Batch ID"
)

batch_id = int(dbutils.widgets.get("batch_id"))

print(f"Selected batch_id for Silver processing: {batch_id}")

In [0]:
bronze_batch_df = (
    spark.table("orders_project.bronze_orders_raw")
    .filter(col("batch_id") == batch_id)
)

display(
    bronze_batch_df.orderBy("order_id", "event_ts")
)

In [0]:
dedup_window = Window.partitionBy("order_id").orderBy(col("event_ts").desc())

silver_updates_df = (
    bronze_batch_df
    .withColumn("row_num", row_number().over(dedup_window))
    .filter(col("row_num") == 1)
    .drop("row_num")
    .select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_amount"),
        col("event_ts").alias("last_event_ts"),
        col("batch_id").alias("last_batch_id"),
        col("source_system"),
        col("source_batch_table"),
        col("bronze_ingestion_ts")
    )
    .withColumn("silver_processed_ts", current_timestamp())
)

display(
    silver_updates_df.orderBy("order_id")
)

In [0]:
silver_table = "orders_project.silver_orders_current"

In [0]:
silver_updates_df.createOrReplaceTempView("silver_updates_temp")

print("Temporary view created: silver_updates_temp")

In [0]:
silver_exists = spark.catalog.tableExists(silver_table)

print(f"Silver table exists: {silver_exists}")

In [0]:
if not silver_exists:
    silver_updates_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(silver_table)

    print(f"Silver table created with batch {batch_id}: {silver_table}")
else:
    print(f"Silver table already exists: {silver_table}")

In [0]:
if silver_exists:
    spark.sql(f"""
        MERGE INTO {silver_table} AS target
        USING silver_updates_temp AS source
        ON target.order_id = source.order_id

        WHEN MATCHED AND source.last_event_ts > target.last_event_ts THEN
          UPDATE SET
            target.customer_id = source.customer_id,
            target.order_status = source.order_status,
            target.order_amount = source.order_amount,
            target.last_event_ts = source.last_event_ts,
            target.last_batch_id = source.last_batch_id,
            target.source_system = source.source_system,
            target.source_batch_table = source.source_batch_table,
            target.bronze_ingestion_ts = source.bronze_ingestion_ts,
            target.silver_processed_ts = source.silver_processed_ts

        WHEN NOT MATCHED THEN
          INSERT (
            order_id,
            customer_id,
            order_status,
            order_amount,
            last_event_ts,
            last_batch_id,
            source_system,
            source_batch_table,
            bronze_ingestion_ts,
            silver_processed_ts
          )
          VALUES (
            source.order_id,
            source.customer_id,
            source.order_status,
            source.order_amount,
            source.last_event_ts,
            source.last_batch_id,
            source.source_system,
            source.source_batch_table,
            source.bronze_ingestion_ts,
            source.silver_processed_ts
          )
    """)

    print(f"MERGE completed for batch {batch_id}.")
else:
    print("MERGE skipped because Silver table was created for the first time.")

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        customer_id,
        order_status,
        order_amount,
        last_event_ts,
        last_batch_id,
        silver_processed_ts
    FROM orders_project.silver_orders_current
    ORDER BY order_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        COUNT(*) AS total_current_orders,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM orders_project.silver_orders_current
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        b.order_id,
        COUNT(*) AS total_events_in_bronze,
        s.order_status AS current_status_in_silver,
        s.last_event_ts
    FROM orders_project.bronze_orders_raw b
    LEFT JOIN orders_project.silver_orders_current s
        ON b.order_id = s.order_id
    GROUP BY
        b.order_id,
        s.order_status,
        s.last_event_ts
    ORDER BY b.order_id
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY orders_project.silver_orders_current
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)